                                Step#1        !!!Start Spark!!!


In [10]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("PatientJourneyBottleneckMiner")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.5.1


                                 Step#2     Load the CSV******************

In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Healthcare Dataset") \
    .getOrCreate()

csv_path = (r"D:\bottleneck-miner\data\healthcare_dataset.csv")  

df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv(csv_path))

print("Total rows:", df.count())
df.printSchema()
df.show(20, truncate=False)

Total rows: 55500
root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Blood Type: string (nullable = true)
 |-- Medical Condition: string (nullable = true)
 |-- Date of Admission: date (nullable = true)
 |-- Doctor: string (nullable = true)
 |-- Hospital: string (nullable = true)
 |-- Insurance Provider: string (nullable = true)
 |-- Billing Amount: double (nullable = true)
 |-- Room Number: integer (nullable = true)
 |-- Admission Type: string (nullable = true)
 |-- Discharge Date: date (nullable = true)
 |-- Medication: string (nullable = true)
 |-- Test Results: string (nullable = true)

+-------------------+---+------+----------+-----------------+-----------------+----------------+---------------------------+------------------+------------------+-----------+--------------+--------------+-----------+------------+
|Name               |Age|Gender|Blood Type|Medical Condition|Date of Admission|Doctor          |Hospital

                       Step#3   ^^^^Clean Data & Build Length-of-Stay Feature^^^^ 

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# Convert column names to snake_case
clean = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])

# Cast types and drop bad rows
clean = (clean
    .withColumn("date_of_admission", F.to_date("date_of_admission"))
    .withColumn("discharge_date", F.to_date("discharge_date"))
    .dropna(subset=["date_of_admission", "discharge_date", "hospital"])
    .withColumn("billing_amount", F.col("billing_amount").cast("double"))
    .withColumn("age", F.col("age").cast("int")))

# ⭐ Core metric: Length of Stay in days
clean = clean.withColumn(
    "length_of_stay",
    F.datediff("discharge_date", "date_of_admission"))

# Keep only realistic stays (0–60 days)
clean = clean.filter((F.col("length_of_stay") >= 0) & (F.col("length_of_stay") <= 60))

# Age groups
clean = clean.withColumn("age_band",
    F.when(F.col("age") < 18, "0-17")
     .when(F.col("age") < 35, "18-34")
     .when(F.col("age") < 55, "35-54")
     .when(F.col("age") < 70, "55-69")
     .otherwise("70+"))

# Cost per day = how expensive each delay day is
clean = clean.withColumn("cost_per_day",
    F.when(F.col("length_of_stay") > 0,
           F.col("billing_amount") / F.col("length_of_stay"))
     .otherwise(F.col("billing_amount")))

clean.cache()
print("Cleaned rows:", clean.count())
clean.select("hospital", "medical_condition", "admission_type",
             "length_of_stay", "cost_per_day", "age_band").show(20)

Cleaned rows: 55500
+--------------------+-----------------+--------------+--------------+------------------+--------+
|            hospital|medical_condition|admission_type|length_of_stay|      cost_per_day|age_band|
+--------------------+-----------------+--------------+--------------+------------------+--------+
|     Sons and Miller|           Cancer|        Urgent|             2| 9428.140652989077|   18-34|
|             Kim Inc|          Obesity|     Emergency|             6| 5607.221214429647|   55-69|
|            Cook PLC|          Obesity|     Emergency|            15|1863.6730719228303|     70+|
|Hernandez Rogers ...|         Diabetes|      Elective|            30|1263.6594136625095|   18-34|
|         White-White|           Cancer|        Urgent|            20| 711.9158906968812|   35-54|
|      Nunez-Humphrey|           Asthma|        Urgent|             4|12036.277737760473|   35-54|
|     Group Middleton|         Diabetes|     Emergency|            12|1631.7393620717442|

                            Step#4  @@@@ Bottleneck Analysis (Spark SQL)@@@@

In [13]:
clean.createOrReplaceTempView("journey")

In [18]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1) Hospital-level bottleneck ranking
# -----------------------------------

hospital_bottleneck = (
    clean.groupBy("hospital")
    .agg(
        F.count("*").alias("admissions"),
        F.round(F.avg("length_of_stay"), 2).alias("avg_los_days"),
        F.round(F.expr("percentile_approx(length_of_stay, 0.5)"), 2).alias("median_los"),
        F.round(F.expr("percentile_approx(length_of_stay, 0.9)"), 2).alias("p90_los"),
        F.round(F.avg("billing_amount"), 2).alias("avg_bill"),
        F.round(F.avg("cost_per_day"), 2).alias("avg_cost_per_day"),
    )
    # Keep only hospitals with enough volume
    .filter(F.col("admissions") >= 20)
    .orderBy(F.col("avg_los_days").desc())
)

hospital_bottleneck.show(10, truncate=False)

hospital_bottleneck.coalesce(1) \
    .write.mode("overwrite") \
    .csv("output/hospital_bottleneck", header=True)


# 2) Condition + admission-type bottlenecks
# -----------------------------------------

cond_admit = (
    clean.groupBy("medical_condition", "admission_type")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("length_of_stay"), 2).alias("avg_los"),
        F.round(F.expr("percentile_approx(length_of_stay, 0.9)"), 2).alias("p90_los"),
    )
    .orderBy(F.col("avg_los").desc())
)

cond_admit.show(30, truncate=False)

cond_admit.coalesce(1) \
    .write.mode("overwrite") \
    .csv("output/condition_admission_bottleneck", header=True)


# 3) Flag top 10% slowest stays per hospital (bottleneck cases)
# -------------------------------------------------------------

w = Window.partitionBy("hospital")

# Compute per-hospital 90th percentile LOS and flag stays above it
flagged = (
    clean
    .withColumn(
        "hospital_p90",
        F.expr("percentile_approx(length_of_stay, 0.9)").over(w)
    )
    .withColumn(
        "is_bottleneck",
        (F.col("length_of_stay") > F.col("hospital_p90")).cast("int")
    )
)

bottleneck_drivers = (
    flagged
    .filter(F.col("is_bottleneck") == 1)
    .groupBy("hospital", "medical_condition", "admission_type")
    .agg(
        F.count("*").alias("delayed_cases"),
        F.round(F.avg("length_of_stay"), 2).alias("avg_los"),
    )
    .orderBy(F.col("delayed_cases").desc())
)

bottleneck_drivers.show(10, truncate=False)

bottleneck_drivers.coalesce(1) \
    .write.mode("overwrite") \
    .csv("output/bottleneck_drivers", header=True)
print("Total hospitals ranked:", hospital_bottleneck.count())
print("Condition combos:", cond_admit.count())
print("Total bottleneck flagged cases:", bottleneck_drivers.count())

+-------------+----------+------------+----------+-------+--------+----------------+
|hospital     |admissions|avg_los_days|median_los|p90_los|avg_bill|avg_cost_per_day|
+-------------+----------+------------+----------+-------+--------+----------------+
|Brown PLC    |22        |21.14       |21        |30     |27801.03|2021.84         |
|Inc Williams |23        |19.04       |20        |29     |31162.5 |1783.04         |
|Smith PLC    |36        |17.64       |21        |27     |28595.12|4183.45         |
|Group Johnson|26        |17.58       |21        |27     |27772.3 |3134.77         |
|Inc Johnson  |26        |17.46       |18        |29     |22589.68|2203.55         |
|Jones Ltd    |22        |17.41       |17        |29     |25142.12|2036.97         |
|LLC Johnson  |30        |17.03       |17        |29     |27214.61|2673.27         |
|Smith Group  |36        |17.0        |17        |28     |22406.42|3776.63         |
|Ltd Smith    |39        |16.51       |17        |28     |25727.3

                    Step#5    Spark MLlib — Predict LoS + Cluster Hospitals

In [1]:
!pip install scikit-learn

     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ------------------- ------------------ 30.7/61.0 kB 640.0 kB/s eta 0:00:01
     ------------------- ------------------ 30.7/61.0 kB 640.0 kB/s eta 0:00:01
     ------------------------- ------------ 41.0/61.0 kB 279.3 kB/s eta 0:00:01
     ------------------------------- ------ 51.2/61.0 kB 260.9 kB/s eta 0:00:01
     -------------------------------------- 61.0/61.0 kB 269.7 kB/s eta 0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.1/8.1 MB 1.1 MB/s eta 0:00:08
   ---------------------------------------- 0.1/8.1 MB 1.3 MB/s eta 0:00:07
    --------------------------------------- 0.1/8.1 MB 819.2 kB/s eta 0:00:10
    --------------------------------------- 0.2/8.1 MB 1.2 MB/s eta 0:00:07
   - -------------------------------------- 0.2/8.1 MB 1.1 MB/s eta 0:00:07
   - -------------------------------------- 0.3/8.1 MB 1.1 MB/s eta 0:00:07
 


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import findspark
findspark.init()

import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.clustering import KMeans
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

print("=" * 60)
print("  STEP 5: Spark MLlib — GBT Regression + KMeans Clustering")
print("=" * 60)

# ================================================================
# PART 1: FEATURE ENGINEERING (fix for low R²)
# Add hospital-level and condition-level LoS statistics as features
# These aggregated signals give GBT something real to learn from
# ================================================================
print("\n[PREP] Engineering aggregated features...")

hosp_stats = (
    clean.groupBy("hospital")
    .agg(
        F.avg("length_of_stay").alias("hosp_avg_los"),
        F.stddev("length_of_stay").alias("hosp_std_los"),
        F.count("*").alias("hosp_volume"),
    )
)

cond_stats = (
    clean.groupBy("medical_condition")
    .agg(F.avg("length_of_stay").alias("cond_avg_los"))
)

admit_stats = (
    clean.groupBy("admission_type")
    .agg(F.avg("length_of_stay").alias("admit_avg_los"))
)

# Join aggregated stats back to patient level
enriched = (
    clean
    .join(hosp_stats, on="hospital", how="left")
    .join(cond_stats, on="medical_condition", how="left")
    .join(admit_stats, on="admission_type", how="left")
    .fillna(
        0,
        subset=[
            "hosp_avg_los",
            "hosp_std_los",
            "hosp_volume",
            "cond_avg_los",
            "admit_avg_los",
        ],
    )
)

print("✅ Feature engineering complete. Enriched rows:", enriched.count())

# ================================================================
# PART 2: GBT REGRESSION — Predict Length of Stay (DELAY)
# ================================================================
print("\n[1/3] Training GBT Regressor (maxIter=100, maxDepth=5)...")

cat_cols = [
    "gender",
    "medical_condition",
    "admission_type",
    "insurance_provider",
    "age_band",
]

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep",
    )
    for c in cat_cols
]
encoders = [
    OneHotEncoder(
        inputCol=c + "_idx",
        outputCol=c + "_oh",
    )
    for c in cat_cols
]

# Include engineered features alongside original ones
num_features = (
    [c + "_oh" for c in cat_cols]
    + [
        "age",
        "billing_amount",
        "hosp_avg_los",
        "hosp_std_los",
        "hosp_volume",
        "cond_avg_los",
        "admit_avg_los",
    ]
)

assembler = VectorAssembler(
    inputCols=num_features,
    outputCol="features",
    handleInvalid="skip",
)

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="length_of_stay",
    maxIter=100,      # was 20 — increased for better convergence
    maxDepth=5,       # controls tree complexity
    stepSize=0.1,     # learning rate
    seed=42,
)

pipe = Pipeline(stages=indexers + encoders + [assembler, gbt])

train, test = enriched.randomSplit([0.8, 0.2], seed=42)
model = pipe.fit(train)
pred = model.transform(test)

rmse = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="rmse",
).evaluate(pred)
r2 = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="r2",
).evaluate(pred)
mae = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="mae",
).evaluate(pred)

print(f"\n✅ GBT Regression Results:")
print(f"   RMSE : {rmse:.2f} days")
print(f"   MAE  : {mae:.2f} days")
print(f"   R²   : {r2:.4f}")
print(f"\n   Interpretation:")
if r2 > 0.15:
    print(f"   ✅ Model explains {r2*100:.1f}% of LoS variance — good for synthetic data.")
elif r2 > 0:
    print(f"   ⚠️  Model explains {r2*100:.1f}% variance. Synthetic data limits signal.")
else:
    print(f"   ⚠️  R²<0: dataset has no real clinical correlation (expected for synthetic data).")
    print(f"       GBT pipeline + feature importance still valid for course demonstration.")

# Feature importance — shows what the model learned
print("\n   Top Feature Importances (GBT):")
gbt_model = model.stages[-1]
feature_names_flat = (
    cat_cols
    + [
        "age",
        "billing_amount",
        "hosp_avg_los",
        "hosp_std_los",
        "hosp_volume",
        "cond_avg_los",
        "admit_avg_los",
    ]
)
importances = gbt_model.featureImportances.toArray()
# Only show non-zero, top 10
import_pairs = sorted(
    zip(feature_names_flat, importances[: len(feature_names_flat)]),
    key=lambda x: x[1],
    reverse=True,
)[:10]
for feat, score in import_pairs:
    print(f"   {feat:<25} {score:.4f}")

# Save LoS prediction sample
(
    pred.select(
        "hospital",
        "medical_condition",
        "admission_type",
        "length_of_stay",
        "prediction",
    )
    .coalesce(1)
    .write.mode("overwrite")
    .csv("output/los_predictions", header=True)
)
print("\n✅ LoS predictions saved → output/los_predictions/")

# ================================================================
# PART 2b: DIAGNOSIS PREDICTION — After Analyzing Delay
# Uses predicted LoS + original features to predict medical_condition
# ================================================================
print("\n[2/3] Predicting diagnosis from delay + features (Logistic Regression)...")

# 1) Get predictions on TRAIN as well, and rename to predicted_los
train_with_pred = model.transform(train).withColumnRenamed("prediction", "predicted_los")
test_with_pred  = pred.withColumnRenamed("prediction", "predicted_los")

# 2) Build classification pipeline (NO medical_condition in input features!)
cls_cat_cols = ["gender", "admission_type", "insurance_provider", "age_band"]

cls_indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx_cls",
        handleInvalid="keep",
    )
    for c in cls_cat_cols
]
cls_encoders = [
    OneHotEncoder(
        inputCol=c + "_idx_cls",
        outputCol=c + "_oh_cls",
    )
    for c in cls_cat_cols
]

cls_feature_cols = (
    [c + "_oh_cls" for c in cls_cat_cols]
    + [
        "age",
        "billing_amount",
        "hosp_avg_los",
        "hosp_std_los",
        "hosp_volume",
        "cond_avg_los",
        "admit_avg_los",
        "predicted_los",  # <-- delay signal from Stage 1
    ]
)

assembler_cls = VectorAssembler(
    inputCols=cls_feature_cols,
    outputCol="diag_features",
    handleInvalid="skip",
)

# Label: index medical_condition (diagnosis)
diag_indexer = StringIndexer(
    inputCol="medical_condition",
    outputCol="diag_label",
    handleInvalid="keep",
)

# Multinomial Logistic Regression for multi-class diagnosis prediction
# (Spark's LogisticRegression supports multinomial via 'family' param).[web:49]
logreg_diag = LogisticRegression(
    featuresCol="diag_features",
    labelCol="diag_label",
    predictionCol="diag_prediction",
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.0,
    family="multinomial",
)

diag_pipe = Pipeline(
    stages=cls_indexers + cls_encoders + [diag_indexer, assembler_cls, logreg_diag]
)

diag_model = diag_pipe.fit(train_with_pred)
diag_pred  = diag_model.transform(test_with_pred)

# 3) Evaluate diagnosis prediction
acc_eval = MulticlassClassificationEvaluator(
    labelCol="diag_label",
    predictionCol="diag_prediction",
    metricName="accuracy",
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="diag_label",
    predictionCol="diag_prediction",
    metricName="f1",
)

diag_acc = acc_eval.evaluate(diag_pred)  # overall accuracy[web:60]
diag_f1  = f1_eval.evaluate(diag_pred)   # macro F1[web:60]

print("\n✅ Diagnosis Prediction Results (after LoS analysis):")
print(f"   Accuracy: {diag_acc:.4f}")
print(f"   F1-score: {diag_f1:.4f}")
print("   (Higher is better — baseline random ~1/num_classes)")

# Save a sample of joint delay + diagnosis predictions
(
    diag_pred.select(
        "hospital",
        "medical_condition",          # true diagnosis
        "diag_prediction",            # predicted diagnosis (indexed)
        "length_of_stay",
        "predicted_los",
        "admission_type",
        "insurance_provider",
    )
    .coalesce(1)
    .write.mode("overwrite")
    .csv("output/diagnosis_from_delay", header=True)
)
print("✅ Diagnosis-from-delay predictions saved → output/diagnosis_from_delay/")

# ================================================================
# PART 3: KMEANS CLUSTERING — with readable labels (unchanged)
# ================================================================
print("\n[3/3] Clustering hospitals (KMeans k=3)...")

hosp_features = (
    hospital_bottleneck
    .na.fill(0)
    .withColumn("avg_los_days",     F.col("avg_los_days").cast(DoubleType()))
    .withColumn("p90_los",          F.col("p90_los").cast(DoubleType()))
    .withColumn("avg_cost_per_day", F.col("avg_cost_per_day").cast(DoubleType()))
    .withColumn("admissions",       F.col("admissions").cast(DoubleType()))
    .cache()
)

va = VectorAssembler(
    inputCols=["avg_los_days", "p90_los", "avg_cost_per_day", "admissions"],
    outputCol="raw",
)
scaler = StandardScaler(
    inputCol="raw",
    outputCol="features",
    withMean=True,
    withStd=True,
)
km = KMeans(
    k=3,
    seed=42,
    featuresCol="features",
    predictionCol="cluster",
)

cluster_model = Pipeline(stages=[va, scaler, km]).fit(hosp_features)
clustered = cluster_model.transform(hosp_features).select(
    "hospital",
    "avg_los_days",
    "median_los",
    "p90_los",
    "avg_cost_per_day",
    "admissions",
    "cluster",
)

# Auto-label clusters: rank by avg_los_days mean per cluster
import pandas as pd

c_pd = clustered.toPandas()
cluster_means = c_pd.groupby("cluster")["avg_los_days"].mean().sort_values()

label_map = {}
keys = cluster_means.index.tolist()
label_map[keys[0]] = "Efficient"      # lowest avg LoS
label_map[keys[1]] = "Average"        # middle
label_map[keys[2]] = "Bottleneck"     # highest avg LoS

c_pd["cluster_label"] = c_pd["cluster"].map(label_map)

print("\n✅ Hospital Cluster Summary:")
summary = (
    c_pd.groupby("cluster_label")
    .agg(
        count=("hospital", "count"),
        avg_los=("avg_los_days", "mean"),
        avg_p90=("p90_los", "mean"),
        avg_cost=("avg_cost_per_day", "mean"),
    )
    .round(2)
)
print(summary.to_string())

print("\n   Bottleneck hospitals (top 5):")
bottleneck_hospitals = (
    c_pd[c_pd["cluster_label"] == "Bottleneck"]
    .sort_values("avg_los_days", ascending=False)
    .head(5)[["hospital", "avg_los_days", "p90_los"]]
)
print(bottleneck_hospitals.to_string(index=False))

print("\n   Efficient hospitals (top 5):")
efficient_hospitals = (
    c_pd[c_pd["cluster_label"] == "Efficient"]
    .sort_values("avg_los_days")
    .head(5)[["hospital", "avg_los_days", "p90_los"]]
)
print(efficient_hospitals.to_string(index=False))

# Save back to Spark and write CSV
clustered_labeled = spark.createDataFrame(c_pd)
(
    clustered_labeled.coalesce(1)
    .write.mode("overwrite")
    .csv("output/hospital_clusters", header=True)
)

print("\n✅ Clusters saved → output/hospital_clusters/")

  STEP 5: Spark MLlib — GBT Regression + KMeans Clustering

[PREP] Engineering aggregated features...
✅ Feature engineering complete. Enriched rows: 55500

[1/3] Training GBT Regressor (maxIter=100, maxDepth=5)...

✅ GBT Regression Results:
   RMSE : 3.94 days
   MAE  : 1.80 days
   R²   : 0.7934

   Interpretation:
   ✅ Model explains 79.3% of LoS variance — good for synthetic data.

   Top Feature Importances (GBT):
   gender                    0.0116
   hosp_avg_los              0.0103
   admit_avg_los             0.0093
   age_band                  0.0071
   age                       0.0070
   admission_type            0.0055
   insurance_provider        0.0054
   hosp_std_los              0.0050
   billing_amount            0.0043
   cond_avg_los              0.0043

✅ LoS predictions saved → output/los_predictions/

[2/3] Predicting diagnosis from delay + features (Logistic Regression)...

✅ Diagnosis Prediction Results (after LoS analysis):
   Accuracy: 0.5605
   F1-score: 0.493

                                   2nd model: Random Forest Regressor

In [30]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# ================================================================
# MODEL 2 — Random Forest Regressor (LoS) + Random Forest Classifier (Diagnosis)
# ================================================================

# ---------- Stage 1: LoS prediction with RandomForestRegressor ----------
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="length_of_stay",
    numTrees=100,
    maxDepth=5,
    seed=42
)

pipe_rf = Pipeline(stages=indexers + encoders + [assembler, rf])
model_rf = pipe_rf.fit(train)
pred_rf  = model_rf.transform(test)

rmse_rf = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="rmse"
).evaluate(pred_rf)
r2_rf   = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="r2"
).evaluate(pred_rf)
mae_rf  = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="mae"
).evaluate(pred_rf)

print(f"\n✅ Random Forest Results (LoS):")
print(f"   RMSE: {rmse_rf:.2f} days | MAE: {mae_rf:.2f} days | R²: {r2_rf:.4f}")
print(f"\n   Model Comparison:")
print(f"   {'Model':<20} {'RMSE':>6} {'MAE':>6} {'R²':>8}")
print(f"   {'GBT Regressor':<20} {rmse:>6.2f} {mae:>6.2f} {r2:>8.4f}")
print(f"   {'Random Forest':<20} {rmse_rf:>6.2f} {mae_rf:>6.2f} {r2_rf:>8.4f}")

# Optional: save LoS predictions from Random Forest as well
(
    pred_rf.select(
        "hospital",
        "medical_condition",
        "admission_type",
        "length_of_stay",
        "prediction"
    )
    .coalesce(1)
    .write.mode("overwrite")
    .csv("output/los_predictions_rf", header=True)
)
print("✅ RF LoS predictions saved → output/los_predictions_rf/")

# ---------- Stage 2: Diagnosis prediction using predicted LoS + features ----------
print("\n[RF Stage 2] Predicting diagnosis from delay + features (RandomForestClassifier)...")

# 1) Get predictions on TRAIN as well, rename to predicted_los
train_rf_with_pred = model_rf.transform(train).withColumnRenamed("prediction", "predicted_los")
test_rf_with_pred  = pred_rf.withColumnRenamed("prediction", "predicted_los")

# 2) Build classification feature set (exclude raw medical_condition as input)
cls_cat_cols = ["gender", "admission_type", "insurance_provider", "age_band"]

cls_indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx_rfcls",
        handleInvalid="keep",
    )
    for c in cls_cat_cols
]
cls_encoders = [
    OneHotEncoder(
        inputCol=c + "_idx_rfcls",
        outputCol=c + "_oh_rfcls",
    )
    for c in cls_cat_cols
]

cls_feature_cols = (
    [c + "_oh_rfcls" for c in cls_cat_cols]
    + [
        "age",
        "billing_amount",
        "hosp_avg_los",
        "hosp_std_los",
        "hosp_volume",
        "cond_avg_los",
        "admit_avg_los",
        "predicted_los",      # <-- delay signal from RF Stage 1
    ]
)

assembler_cls_rf = VectorAssembler(
    inputCols=cls_feature_cols,
    outputCol="rf_diag_features",
    handleInvalid="skip",
)

diag_indexer_rf = StringIndexer(
    inputCol="medical_condition",
    outputCol="rf_diag_label",
    handleInvalid="keep",
)

rf_cls = RandomForestClassifier(
    featuresCol="rf_diag_features",
    labelCol="rf_diag_label",
    predictionCol="rf_diag_prediction",
    numTrees=100,
    maxDepth=5,
    seed=42,
)

diag_pipe_rf = Pipeline(
    stages=cls_indexers + cls_encoders + [diag_indexer_rf, assembler_cls_rf, rf_cls]
)

diag_model_rf = diag_pipe_rf.fit(train_rf_with_pred)
diag_pred_rf  = diag_model_rf.transform(test_rf_with_pred)

# 3) Evaluate diagnosis prediction (accuracy + F1)[web:60]
acc_eval_rf = MulticlassClassificationEvaluator(
    labelCol="rf_diag_label",
    predictionCol="rf_diag_prediction",
    metricName="accuracy",
)
f1_eval_rf = MulticlassClassificationEvaluator(
    labelCol="rf_diag_label",
    predictionCol="rf_diag_prediction",
    metricName="f1",
)

diag_acc_rf = acc_eval_rf.evaluate(diag_pred_rf)
diag_f1_rf  = f1_eval_rf.evaluate(diag_pred_rf)

print("\n✅ RF Diagnosis Prediction Results (after LoS analysis):")
print(f"   Accuracy: {diag_acc_rf:.4f}")
print(f"   F1-score: {diag_f1_rf:.4f}")
print("   (Higher is better — baseline random ≈ 1 / num_conditions)")

# Save a sample of diagnosis-from-delay predictions for RF
(
    diag_pred_rf.select(
        "hospital",
        "medical_condition",     # true diagnosis
        "rf_diag_prediction",    # predicted diagnosis (indexed)
        "length_of_stay",
        "predicted_los",
        "admission_type",
        "insurance_provider",
    )
    .coalesce(1)
    .write.mode("overwrite")
    .csv("output/diagnosis_from_delay_rf", header=True)
)
print("✅ RF diagnosis-from-delay predictions saved → output/diagnosis_from_delay_rf/")


✅ Random Forest Results (LoS):
   RMSE: 4.46 days | MAE: 3.15 days | R²: 0.7356

   Model Comparison:
   Model                  RMSE    MAE       R²
   GBT Regressor          3.94   1.80   0.7934
   Random Forest          4.46   3.15   0.7356
✅ RF LoS predictions saved → output/los_predictions_rf/

[RF Stage 2] Predicting diagnosis from delay + features (RandomForestClassifier)...

✅ RF Diagnosis Prediction Results (after LoS analysis):
   Accuracy: 1.0000
   F1-score: 1.0000
   (Higher is better — baseline random ≈ 1 / num_conditions)
✅ RF diagnosis-from-delay predictions saved → output/diagnosis_from_delay_rf/


                         3rd Model Linear Regression

In [31]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator

# ================================================================
# MODEL 3 — Linear Regression (LoS) + Logistic Regression (Diagnosis)
# ================================================================

# ---------- Stage 1: LoS prediction with LinearRegression ----------
lr = LinearRegression(
    featuresCol="features",
    labelCol="length_of_stay",
    maxIter=100,
    regParam=0.1,        # L2 regularization
    elasticNetParam=0.0
)

pipe_lr = Pipeline(stages=indexers + encoders + [assembler, lr])
model_lr = pipe_lr.fit(train)
pred_lr  = model_lr.transform(test)

rmse_lr = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="rmse"
).evaluate(pred_lr)
r2_lr   = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="r2"
).evaluate(pred_lr)
mae_lr  = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="mae"
).evaluate(pred_lr)

print(f"\n✅ Linear Regression Results (LoS):")
print(f"   RMSE: {rmse_lr:.2f} | MAE: {mae_lr:.2f} | R²: {r2_lr:.4f}")

# ---------- Stage 2: Diagnosis prediction from delay + features ----------
print("\n[LR Stage 2] Predicting diagnosis from delay + features (LogisticRegression)...")

# 1) Get predictions on TRAIN as well, rename to predicted_los
train_lr_with_pred = model_lr.transform(train).withColumnRenamed("prediction", "predicted_los")
test_lr_with_pred  = pred_lr.withColumnRenamed("prediction", "predicted_los")

# 2) Classification feature set (no raw medical_condition as input)
cls_cat_cols_lr = ["gender", "admission_type", "insurance_provider", "age_band"]

cls_indexers_lr = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx_lrcls",
        handleInvalid="keep",
    )
    for c in cls_cat_cols_lr
]
cls_encoders_lr = [
    OneHotEncoder(
        inputCol=c + "_idx_lrcls",
        outputCol=c + "_oh_lrcls",
    )
    for c in cls_cat_cols_lr
]

cls_feature_cols_lr = (
    [c + "_oh_lrcls" for c in cls_cat_cols_lr]
    + [
        "age",
        "billing_amount",
        "hosp_avg_los",
        "hosp_std_los",
        "hosp_volume",
        "cond_avg_los",
        "admit_avg_los",
        "predicted_los",     # <-- delay signal from LR Stage 1
    ]
)

assembler_cls_lr = VectorAssembler(
    inputCols=cls_feature_cols_lr,
    outputCol="lr_diag_features",
    handleInvalid="skip",
)

diag_indexer_lr = StringIndexer(
    inputCol="medical_condition",
    outputCol="lr_diag_label",
    handleInvalid="keep",
)

# Multinomial Logistic Regression for multi-class diagnosis prediction[web:49]
logreg_diag_lr = LogisticRegression(
    featuresCol="lr_diag_features",
    labelCol="lr_diag_label",
    predictionCol="lr_diag_prediction",
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.0,
    family="multinomial",
)

diag_pipe_lr = Pipeline(
    stages=cls_indexers_lr + cls_encoders_lr + [diag_indexer_lr, assembler_cls_lr, logreg_diag_lr]
)

diag_model_lr = diag_pipe_lr.fit(train_lr_with_pred)
diag_pred_lr  = diag_model_lr.transform(test_lr_with_pred)

# 3) Evaluate diagnosis prediction (accuracy + F1)[web:60]
acc_eval_lr = MulticlassClassificationEvaluator(
    labelCol="lr_diag_label",
    predictionCol="lr_diag_prediction",
    metricName="accuracy",
)
f1_eval_lr = MulticlassClassificationEvaluator(
    labelCol="lr_diag_label",
    predictionCol="lr_diag_prediction",
    metricName="f1",
)

diag_acc_lr = acc_eval_lr.evaluate(diag_pred_lr)
diag_f1_lr  = f1_eval_lr.evaluate(diag_pred_lr)

print("\n✅ LR Diagnosis Prediction Results (after LoS analysis):")
print(f"   Accuracy: {diag_acc_lr:.4f}")
print(f"   F1-score: {diag_f1_lr:.4f}")
print("   (Higher is better — random baseline ≈ 1 / num_conditions)")

# Save joint delay + diagnosis predictions
(
    diag_pred_lr.select(
        "hospital",
        "medical_condition",        # true diagnosis
        "lr_diag_prediction",       # predicted diagnosis (indexed)
        "length_of_stay",
        "predicted_los",
        "admission_type",
        "insurance_provider",
    )
    .coalesce(1)
    .write.mode("overwrite")
    .csv("output/diagnosis_from_delay_lr", header=True)
)
print("✅ LR diagnosis-from-delay predictions saved → output/diagnosis_from_delay_lr/")


✅ Linear Regression Results (LoS):
   RMSE: 3.89 | MAE: 1.80 | R²: 0.7988

[LR Stage 2] Predicting diagnosis from delay + features (LogisticRegression)...

✅ LR Diagnosis Prediction Results (after LoS analysis):
   Accuracy: 0.5608
   F1-score: 0.4943
   (Higher is better — random baseline ≈ 1 / num_conditions)
✅ LR diagnosis-from-delay predictions saved → output/diagnosis_from_delay_lr/


                              4th Model  Decision Tree Regressor

In [32]:
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# ================================================================
# MODEL 4: Decision Tree Regressor (LoS)
# ================================================================
dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="length_of_stay",
    maxDepth=5,
    seed=42
)
pipe_dt = Pipeline(stages=indexers + encoders + [assembler, dt])
model_dt = pipe_dt.fit(train)
pred_dt  = model_dt.transform(test)

rmse_dt = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="rmse"
).evaluate(pred_dt)
r2_dt   = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="r2"
).evaluate(pred_dt)
mae_dt  = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="mae"
).evaluate(pred_dt)
print(f"\nDecision Tree — RMSE: {rmse_dt:.2f} | MAE: {mae_dt:.2f} | R²: {r2_dt:.4f}")

# ================================================================
# FIXED Feature Importance Extraction for Decision Tree (unchanged)
# ================================================================
dt_model = model_dt.stages[-1]
dt_importances_full = dt_model.featureImportances.toArray()

# Get actual vector size from a transformed sample
assembled_sample = (
    Pipeline(stages=indexers + encoders + [assembler])
    .fit(enriched)
    .transform(enriched.limit(1))
)
feature_size = len(assembled_sample.select("features").first()[0])

# Numeric features are always appended at the END of the vector
numeric_names = [
    "age",
    "billing_amount",
    "hosp_avg_los",
    "hosp_std_los",
    "hosp_volume",
    "cond_avg_los",
    "admit_avg_los",
]
num_start = feature_size - len(numeric_names)

print("\nDecision Tree — Numeric Feature Importances:")
for i, name in enumerate(numeric_names):
    score = dt_importances_full[num_start + i]
    print(f"  {name:<25} {score:.4f}")

# Top indices across ALL features (includes OHE categorical slots)
top_indices = dt_importances_full.argsort()[::-1][:10]
print("\nDecision Tree — Top 10 Feature Slots (all features):")
for i in top_indices:
    if dt_importances_full[i] > 0:
        print(f"  Index {i:<5} importance: {dt_importances_full[i]:.4f}")

# ================================================================
# NEW: Stage 2 — Diagnosis prediction using DecisionTreeClassifier
#       (uses predicted LoS from Stage 1 as an input feature)
# ================================================================
print("\n[DT Stage 2] Predicting diagnosis from delay + features (DecisionTreeClassifier)...")

# 1) Get predictions on TRAIN as well, rename regression prediction to predicted_los
train_dt_with_pred = model_dt.transform(train).withColumnRenamed("prediction", "predicted_los")
test_dt_with_pred  = pred_dt.withColumnRenamed("prediction", "predicted_los")

# 2) Build classification feature set (exclude raw medical_condition as input)
cls_cat_cols_dt = ["gender", "admission_type", "insurance_provider", "age_band"]

cls_indexers_dt = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx_dtcls",
        handleInvalid="keep",
    )
    for c in cls_cat_cols_dt
]
cls_encoders_dt = [
    OneHotEncoder(
        inputCol=c + "_idx_dtcls",
        outputCol=c + "_oh_dtcls",
    )
    for c in cls_cat_cols_dt
]

cls_feature_cols_dt = (
    [c + "_oh_dtcls" for c in cls_cat_cols_dt]
    + [
        "age",
        "billing_amount",
        "hosp_avg_los",
        "hosp_std_los",
        "hosp_volume",
        "cond_avg_los",
        "admit_avg_los",
        "predicted_los",  # <-- delay signal from DT regression
    ]
)

assembler_cls_dt = VectorAssembler(
    inputCols=cls_feature_cols_dt,
    outputCol="dt_diag_features",
    handleInvalid="skip",
)

diag_indexer_dt = StringIndexer(
    inputCol="medical_condition",
    outputCol="dt_diag_label",
    handleInvalid="keep",
)

dt_cls = DecisionTreeClassifier(
    featuresCol="dt_diag_features",
    labelCol="dt_diag_label",
    predictionCol="dt_diag_prediction",
    maxDepth=5,
    seed=42,
)

diag_pipe_dt = Pipeline(
    stages=cls_indexers_dt + cls_encoders_dt + [diag_indexer_dt, assembler_cls_dt, dt_cls]
)

diag_model_dt = diag_pipe_dt.fit(train_dt_with_pred)
diag_pred_dt  = diag_model_dt.transform(test_dt_with_pred)

# 3) Evaluate diagnosis prediction (accuracy + F1)[web:60]
acc_eval_dt = MulticlassClassificationEvaluator(
    labelCol="dt_diag_label",
    predictionCol="dt_diag_prediction",
    metricName="accuracy",
)
f1_eval_dt = MulticlassClassificationEvaluator(
    labelCol="dt_diag_label",
    predictionCol="dt_diag_prediction",
    metricName="f1",
)

diag_acc_dt = acc_eval_dt.evaluate(diag_pred_dt)
diag_f1_dt  = f1_eval_dt.evaluate(diag_pred_dt)

print("\n✅ DT Diagnosis Prediction Results (after LoS analysis):")
print(f"   Accuracy: {diag_acc_dt:.4f}")
print(f"   F1-score: {diag_f1_dt:.4f}")
print("   (Higher is better — random baseline ≈ 1 / num_conditions)")

# Save joint delay + diagnosis predictions for DT
(
    diag_pred_dt.select(
        "hospital",
        "medical_condition",       # true diagnosis
        "dt_diag_prediction",      # predicted diagnosis (indexed)
        "length_of_stay",
        "predicted_los",
        "admission_type",
        "insurance_provider",
    )
    .coalesce(1)
    .write.mode("overwrite")
    .csv("output/diagnosis_from_delay_dt", header=True)
)
print("✅ DT diagnosis-from-delay predictions saved → output/diagnosis_from_delay_dt/")


Decision Tree — RMSE: 3.90 | MAE: 1.83 | R²: 0.7980

Decision Tree — Numeric Feature Importances:
  age                       0.0000
  billing_amount            0.0001
  hosp_avg_los              0.9998
  hosp_std_los              0.0000
  hosp_volume               0.0000
  cond_avg_los              0.0000
  admit_avg_los             0.0000

Decision Tree — Top 10 Feature Slots (all features):
  Index 23    importance: 0.9998
  Index 16    importance: 0.0001
  Index 22    importance: 0.0001
  Index 24    importance: 0.0000

[DT Stage 2] Predicting diagnosis from delay + features (DecisionTreeClassifier)...

✅ DT Diagnosis Prediction Results (after LoS analysis):
   Accuracy: 1.0000
   F1-score: 1.0000
   (Higher is better — random baseline ≈ 1 / num_conditions)
✅ DT diagnosis-from-delay predictions saved → output/diagnosis_from_delay_dt/


                     5th Model :GBT + CrossValidator

In [33]:
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# ================================================================
# MODEL 5: GBT + CrossValidation (LoS) → Random Forest Classifier (Diagnosis)
# ================================================================

# ---------- Stage 1: Tuned GBT Regressor with CrossValidator ----------
gbt_cv = GBTRegressor(
    featuresCol="features",
    labelCol="length_of_stay",
    seed=42
)

pipe_cv = Pipeline(stages=indexers + encoders + [assembler, gbt_cv])

paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt_cv.maxIter,  [50, 100])
    .addGrid(gbt_cv.maxDepth, [3, 5])
    .addGrid(gbt_cv.stepSize, [0.05, 0.1])
    .build()
)

evaluator = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="rmse"
)

cv = CrossValidator(
    estimator=pipe_cv,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,      # 3-fold cross validation
    seed=42
)

cv_model = cv.fit(train)
pred_cv  = cv_model.transform(test)

rmse_cv = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="rmse"
).evaluate(pred_cv)
r2_cv   = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="r2"
).evaluate(pred_cv)
mae_cv  = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="mae"
).evaluate(pred_cv)

print(f"\nGBT + CrossVal — RMSE: {rmse_cv:.2f} | MAE: {mae_cv:.2f} | R²: {r2_cv:.4f}")
print("Best params found by CrossValidator ↑")

# Optional: save tuned LoS predictions
(
    pred_cv.select(
        "hospital",
        "medical_condition",
        "admission_type",
        "length_of_stay",
        "prediction"
    )
    .coalesce(1)
    .write.mode("overwrite")
    .csv("output/los_predictions_gbt_cv", header=True)
)
print("✅ Tuned GBT LoS predictions saved → output/los_predictions_gbt_cv/")

# ---------- Stage 2: Diagnosis prediction using RFClassifier ----------
print("\n[GBT+CV Stage 2] Predicting diagnosis from delay + features (RandomForestClassifier)...")

# 1) Get predictions on TRAIN as well, rename prediction→predicted_los
train_cv_with_pred = cv_model.transform(train).withColumnRenamed("prediction", "predicted_los")
test_cv_with_pred  = pred_cv.withColumnRenamed("prediction", "predicted_los")

# 2) Classification feature set (exclude raw medical_condition as input)
cls_cat_cols_cv = ["gender", "admission_type", "insurance_provider", "age_band"]

cls_indexers_cv = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx_cvcls",
        handleInvalid="keep",
    )
    for c in cls_cat_cols_cv
]
cls_encoders_cv = [
    OneHotEncoder(
        inputCol=c + "_idx_cvcls",
        outputCol=c + "_oh_cvcls",
    )
    for c in cls_cat_cols_cv
]

cls_feature_cols_cv = (
    [c + "_oh_cvcls" for c in cls_cat_cols_cv]
    + [
        "age",
        "billing_amount",
        "hosp_avg_los",
        "hosp_std_los",
        "hosp_volume",
        "cond_avg_los",
        "admit_avg_los",
        "predicted_los",      # <-- tuned delay signal from Stage 1
    ]
)

assembler_cls_cv = VectorAssembler(
    inputCols=cls_feature_cols_cv,
    outputCol="cv_diag_features",
    handleInvalid="skip",
)

diag_indexer_cv = StringIndexer(
    inputCol="medical_condition",
    outputCol="cv_diag_label",
    handleInvalid="keep",
)

rf_cls_cv = RandomForestClassifier(
    featuresCol="cv_diag_features",
    labelCol="cv_diag_label",
    predictionCol="cv_diag_prediction",
    numTrees=100,
    maxDepth=5,
    seed=42,
)

diag_pipe_cv = Pipeline(
    stages=cls_indexers_cv + cls_encoders_cv + [diag_indexer_cv, assembler_cls_cv, rf_cls_cv]
)

diag_model_cv = diag_pipe_cv.fit(train_cv_with_pred)
diag_pred_cv  = diag_model_cv.transform(test_cv_with_pred)

# 3) Evaluate diagnosis prediction (accuracy + F1)[web:60]
acc_eval_cv = MulticlassClassificationEvaluator(
    labelCol="cv_diag_label",
    predictionCol="cv_diag_prediction",
    metricName="accuracy",
)
f1_eval_cv = MulticlassClassificationEvaluator(
    labelCol="cv_diag_label",
    predictionCol="cv_diag_prediction",
    metricName="f1",
)

diag_acc_cv = acc_eval_cv.evaluate(diag_pred_cv)
diag_f1_cv  = f1_eval_cv.evaluate(diag_pred_cv)

print("\n✅ GBT+CV → RF Diagnosis Prediction Results (after LoS analysis):")
print(f"   Accuracy: {diag_acc_cv:.4f}")
print(f"   F1-score: {diag_f1_cv:.4f}")
print("   (Higher is better — random baseline ≈ 1 / num_conditions)")

# Save joint delay + diagnosis predictions for tuned pipeline
(
    diag_pred_cv.select(
        "hospital",
        "medical_condition",       # true diagnosis
        "cv_diag_prediction",      # predicted diagnosis (indexed)
        "length_of_stay",
        "predicted_los",
        "admission_type",
        "insurance_provider",
    )
    .coalesce(1)
    .write.mode("overwrite")
    .csv("output/diagnosis_from_delay_gbt_cv", header=True)
)
print("✅ GBT+CV diagnosis-from-delay predictions saved → output/diagnosis_from_delay_gbt_cv/")


GBT + CrossVal — RMSE: 3.90 | MAE: 1.90 | R²: 0.7975
Best params found by CrossValidator ↑
✅ Tuned GBT LoS predictions saved → output/los_predictions_gbt_cv/

[GBT+CV Stage 2] Predicting diagnosis from delay + features (RandomForestClassifier)...

✅ GBT+CV → RF Diagnosis Prediction Results (after LoS analysis):
   Accuracy: 1.0000
   F1-score: 1.0000
   (Higher is better — random baseline ≈ 1 / num_conditions)
✅ GBT+CV diagnosis-from-delay predictions saved → output/diagnosis_from_delay_gbt_cv/


                   6th Model:  Generalized Linear Regression (GLM) with Poisson family

In [35]:
from pyspark.ml.regression import GeneralizedLinearRegression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# ================================================================
# MODEL 6: GLM Poisson (LoS) + Logistic Regression (Diagnosis)
# ================================================================

# ---------- Stage 1: LoS prediction with GLM Poisson ----------
glm = GeneralizedLinearRegression(
    featuresCol="features",
    labelCol="length_of_stay",
    family="poisson",      # count data distribution
    link="log",
    maxIter=100,
    regParam=0.1
)
pipe_glm = Pipeline(stages=indexers + encoders + [assembler, glm])
model_glm = pipe_glm.fit(train)
pred_glm  = model_glm.transform(test)

rmse_glm = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="rmse"
).evaluate(pred_glm)
r2_glm   = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="r2"
).evaluate(pred_glm)
mae_glm  = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="mae"
).evaluate(pred_glm)

print(f"\nGLM Poisson — RMSE: {rmse_glm:.2f} | MAE: {mae_glm:.2f} | R²: {r2_glm:.4f}")

# Optional: save LoS predictions from GLM
(
    pred_glm.select(
        "hospital",
        "medical_condition",
        "admission_type",
        "length_of_stay",
        "prediction"
    )
    .coalesce(1)
    .write.mode("overwrite")
    .csv("output/los_predictions_glm", header=True)
)
print("✅ GLM LoS predictions saved → output/los_predictions_glm/")

# ---------- Stage 2: Diagnosis prediction from delay + features ----------
print("\n[GLM Stage 2] Predicting diagnosis from delay + features (LogisticRegression)...")

# 1) Get predictions on TRAIN as well, rename prediction→predicted_los
train_glm_with_pred = model_glm.transform(train).withColumnRenamed("prediction", "predicted_los")
test_glm_with_pred  = pred_glm.withColumnRenamed("prediction", "predicted_los")

# 2) Classification feature set (exclude raw medical_condition as input)
cls_cat_cols_glm = ["gender", "admission_type", "insurance_provider", "age_band"]

cls_indexers_glm = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx_glmcls",
        handleInvalid="keep",
    )
    for c in cls_cat_cols_glm
]
cls_encoders_glm = [
    OneHotEncoder(
        inputCol=c + "_idx_glmcls",
        outputCol=c + "_oh_glmcls",
    )
    for c in cls_cat_cols_glm
]

cls_feature_cols_glm = (
    [c + "_oh_glmcls" for c in cls_cat_cols_glm]
    + [
        "age",
        "billing_amount",
        "hosp_avg_los",
        "hosp_std_los",
        "hosp_volume",
        "cond_avg_los",
        "admit_avg_los",
        "predicted_los",     # <-- delay signal from GLM Stage 1
    ]
)

assembler_cls_glm = VectorAssembler(
    inputCols=cls_feature_cols_glm,
    outputCol="glm_diag_features",
    handleInvalid="skip",
)

diag_indexer_glm = StringIndexer(
    inputCol="medical_condition",
    outputCol="glm_diag_label",
    handleInvalid="keep",
)

# Multinomial Logistic Regression for multi-class diagnosis prediction[web:49]
logreg_diag_glm = LogisticRegression(
    featuresCol="glm_diag_features",
    labelCol="glm_diag_label",
    predictionCol="glm_diag_prediction",
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.0,
    family="multinomial",
)

diag_pipe_glm = Pipeline(
    stages=cls_indexers_glm + cls_encoders_glm + [diag_indexer_glm, assembler_cls_glm, logreg_diag_glm]
)

diag_model_glm = diag_pipe_glm.fit(train_glm_with_pred)
diag_pred_glm  = diag_model_glm.transform(test_glm_with_pred)

# 3) Evaluate diagnosis prediction (accuracy + F1)[web:60]
acc_eval_glm = MulticlassClassificationEvaluator(
    labelCol="glm_diag_label",
    predictionCol="glm_diag_prediction",
    metricName="accuracy",
)
f1_eval_glm = MulticlassClassificationEvaluator(
    labelCol="glm_diag_label",
    predictionCol="glm_diag_prediction",
    metricName="f1",
)

diag_acc_glm = acc_eval_glm.evaluate(diag_pred_glm)
diag_f1_glm  = f1_eval_glm.evaluate(diag_pred_glm)

print("\n✅ GLM → Logistic Regression Diagnosis Results (after LoS analysis):")
print(f"   Accuracy: {diag_acc_glm:.4f}")
print(f"   F1-score: {diag_f1_glm:.4f}")
print("   (Higher is better — random baseline ≈ 1 / num_conditions)")

# Save joint delay + diagnosis predictions for GLM pipeline
(
    diag_pred_glm.select(
        "hospital",
        "medical_condition",        # true diagnosis
        "glm_diag_prediction",      # predicted diagnosis (indexed)
        "length_of_stay",
        "predicted_los",
        "admission_type",
        "insurance_provider",
    )
    .coalesce(1)
    .write.mode("overwrite")
    .csv("output/diagnosis_from_delay_glm", header=True)
)
print("✅ GLM diagnosis-from-delay predictions saved → output/diagnosis_from_delay_glm/")


GLM Poisson — RMSE: 4.36 | MAE: 3.10 | R²: 0.7470
✅ GLM LoS predictions saved → output/los_predictions_glm/

[GLM Stage 2] Predicting diagnosis from delay + features (LogisticRegression)...

✅ GLM → Logistic Regression Diagnosis Results (after LoS analysis):
   Accuracy: 0.5612
   F1-score: 0.4946
   (Higher is better — random baseline ≈ 1 / num_conditions)
✅ GLM diagnosis-from-delay predictions saved → output/diagnosis_from_delay_glm/


                             7th Model :Factorization Machines Regressor



In [37]:
from pyspark.ml.regression import FMRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import MinMaxScaler, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import numpy as np

# ================================================================
# MODEL 7: Factorization Machines Regressor (LoS)
# ================================================================
fm_scaler = MinMaxScaler(inputCol="features", outputCol="scaled_features")

fm = FMRegressor(
    featuresCol="scaled_features",
    labelCol="length_of_stay",
    stepSize=0.001,
    maxIter=100,
    seed=42,
    factorSize=8
)

pipe_fm = Pipeline(stages=indexers + encoders + [assembler, fm_scaler, fm])
model_fm = pipe_fm.fit(train)
pred_fm  = model_fm.transform(test)

rmse_fm = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="rmse"
).evaluate(pred_fm)
r2_fm   = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="r2"
).evaluate(pred_fm)
mae_fm  = RegressionEvaluator(
    labelCol="length_of_stay",
    metricName="mae"
).evaluate(pred_fm)

print(f"\nFM Regressor — RMSE: {rmse_fm:.2f} | MAE: {mae_fm:.2f} | R²: {r2_fm:.4f}")

# ================================================================
# PAIRWISE INTERACTION ANALYSIS (unchanged)
# ================================================================
fm_model_stage = model_fm.stages[-1]
linear_weights  = fm_model_stage.linear.toArray()

numeric_names = [
    "age", "billing_amount", "hosp_avg_los",
    "hosp_std_los", "hosp_volume", "cond_avg_los", "admit_avg_los",
]

assembled_sample = (
    Pipeline(stages=indexers + encoders + [assembler])
    .fit(enriched)
    .transform(enriched.limit(1))
)
feature_size = len(assembled_sample.select("features").first()[0])
num_start = feature_size - len(numeric_names)

print("\n--- FM Linear Weights (direct effect of each feature on LoS) ---")
numeric_weights = []
for i, name in enumerate(numeric_names):
    w = linear_weights[num_start + i]
    numeric_weights.append((name, w))

numeric_weights.sort(key=lambda x: abs(x[1]), reverse=True)
for name, w in numeric_weights:
    direction = "↑ increases LoS" if w > 0 else "↓ decreases LoS"
    print(f"  {name:<25} weight: {w:+.4f}  {direction}")

factors = fm_model_stage.factors.toArray()

print("\n--- FM Pairwise Interaction Strengths (numeric features) ---")
print(f"  {'Feature A':<22} × {'Feature B':<22}  Interaction Score")
print(f"  {'-'*65}")

interactions = []
for i in range(len(numeric_names)):
    for j in range(i + 1, len(numeric_names)):
        fi = factors[num_start + i]
        fj = factors[num_start + j]
        score = float(np.dot(fi, fj))
        interactions.append((numeric_names[i], numeric_names[j], score))

interactions.sort(key=lambda x: abs(x[2]), reverse=True)
for a, b, score in interactions[:10]:
    strength = (
        "STRONG"   if abs(score) > 0.01
        else "moderate" if abs(score) > 0.001
        else "weak"
    )
    print(f"  {a:<22} × {b:<22}  {score:+.6f}  ({strength})")

print("\n  Interpretation: Positive score = features jointly INCREASE LoS")
print("                  Negative score = features have OPPOSING effects on LoS")

# ================================================================
# STAGE 2 — Diagnosis prediction with RandomForestClassifier
# FMClassifier is binary-only in PySpark MLlib, so RF handles
# the 6-class medical_condition prediction reliably.[web:49]
# ================================================================
print("\n[FM Stage 2] Predicting diagnosis from delay + features (RandomForestClassifier)...")

# Step 1: rename regression 'prediction' → 'predicted_los' on BOTH splits
# Drop the scaled_features column first to avoid VectorAssembler conflicts
train_fm_pred = (
    model_fm.transform(train)
    .withColumnRenamed("prediction", "predicted_los")
    .drop("scaled_features")        # avoid column name clash in Stage 2
)
test_fm_pred = (
    pred_fm
    .withColumnRenamed("prediction", "predicted_los")
    .drop("scaled_features")        # same drop on test
)

# Step 2: fresh encoding with unique suffix _fm2 to avoid any column conflicts
cls_cat_cols_fm2 = ["gender", "admission_type", "insurance_provider", "age_band"]

cls_indexers_fm2 = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx_fm2",
        handleInvalid="keep",
    )
    for c in cls_cat_cols_fm2
]
cls_encoders_fm2 = [
    OneHotEncoder(
        inputCol=c + "_idx_fm2",
        outputCol=c + "_oh_fm2",
    )
    for c in cls_cat_cols_fm2
]

cls_feature_cols_fm2 = (
    [c + "_oh_fm2" for c in cls_cat_cols_fm2]
    + [
        "age",
        "billing_amount",
        "hosp_avg_los",
        "hosp_std_los",
        "hosp_volume",
        "cond_avg_los",
        "admit_avg_los",
        "predicted_los",    # <-- delay signal from FM Stage 1
    ]
)

assembler_fm2 = VectorAssembler(
    inputCols=cls_feature_cols_fm2,
    outputCol="fm2_diag_features",
    handleInvalid="skip",
)

diag_indexer_fm2 = StringIndexer(
    inputCol="medical_condition",
    outputCol="fm2_diag_label",
    handleInvalid="keep",
)

# RandomForestClassifier handles 6-class multiclass natively
rf_cls_fm2 = RandomForestClassifier(
    featuresCol="fm2_diag_features",
    labelCol="fm2_diag_label",
    predictionCol="fm2_diag_prediction",
    numTrees=100,
    maxDepth=5,
    seed=42,
)

diag_pipe_fm2 = Pipeline(
    stages=cls_indexers_fm2 + cls_encoders_fm2
         + [diag_indexer_fm2, assembler_fm2, rf_cls_fm2]
)

# Step 3: fit on train, transform on test — no more FMClassifier crash
diag_model_fm2 = diag_pipe_fm2.fit(train_fm_pred)
diag_pred_fm2  = diag_model_fm2.transform(test_fm_pred)

# Step 4: evaluate
acc_eval_fm2 = MulticlassClassificationEvaluator(
    labelCol="fm2_diag_label",
    predictionCol="fm2_diag_prediction",
    metricName="accuracy",
)
f1_eval_fm2 = MulticlassClassificationEvaluator(
    labelCol="fm2_diag_label",
    predictionCol="fm2_diag_prediction",
    metricName="f1",
)

diag_acc_fm2 = acc_eval_fm2.evaluate(diag_pred_fm2)
diag_f1_fm2  = f1_eval_fm2.evaluate(diag_pred_fm2)

print("\n✅ FM → RF Diagnosis Prediction Results (after LoS analysis):")
print(f"   Accuracy: {diag_acc_fm2:.4f}")
print(f"   F1-score: {diag_f1_fm2:.4f}")
print("   (Higher is better — random baseline ≈ 1 / 6 conditions ≈ 0.167)")

# Step 5: save
(
    diag_pred_fm2.select(
        "hospital",
        "medical_condition",       # true diagnosis
        "fm2_diag_prediction",     # predicted diagnosis (indexed)
        "length_of_stay",
        "predicted_los",
        "admission_type",
        "insurance_provider",
    )
    .coalesce(1)
    .write.mode("overwrite")
    .csv("output/diagnosis_from_delay_fm", header=True)
)
print("✅ FM diagnosis-from-delay predictions saved → output/diagnosis_from_delay_fm/")


FM Regressor — RMSE: 13.89 | MAE: 11.50 | R²: -1.5668

--- FM Linear Weights (direct effect of each feature on LoS) ---
  hosp_avg_los              weight: +0.0964  ↑ increases LoS
  age                       weight: +0.0954  ↑ increases LoS
  billing_amount            weight: +0.0954  ↑ increases LoS
  admit_avg_los             weight: +0.0952  ↑ increases LoS
  cond_avg_los              weight: +0.0952  ↑ increases LoS
  hosp_std_los              weight: +0.0950  ↑ increases LoS
  hosp_volume               weight: +0.0949  ↑ increases LoS

--- FM Pairwise Interaction Strengths (numeric features) ---
  Feature A              × Feature B               Interaction Score
  -----------------------------------------------------------------
  age                    × hosp_avg_los            +0.153161  (STRONG)
  age                    × admit_avg_los           +0.151659  (STRONG)
  age                    × billing_amount          +0.150900  (STRONG)
  age                    × cond_avg_los 

                       -----------visulizations------------

In [18]:
!pip install plotly dash

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB 1.4 MB/s eta 0:00:08
   ---------------------------------------- 0.0/9.9 MB 1.4 MB/s eta 0:00:08
   ---------------------------------------- 0.1/9.9 MB 409.6 kB/s eta 0:00:25
   ---------------------------------------- 0.1/9.9 MB 409.6 kB/s eta 0:00:25
   ---------------------------------------- 0.1/9.9 MB 409.6 kB/s eta 0:00:25
   ---------------------------------------- 0.1/9.9 MB 409.6 kB/s eta 0:00:25
    --------------------------------------- 0.2/9.9 MB 622.7 kB/s eta 0:00:16
    --------------------------------------- 0.2/9.9 MB 622.7 kB/s eta 0:00:16
    --------------------------------------- 0.2/9.9 MB 479.2 kB/s eta 0:00:21
    --------------------------------------- 0.2/9.9 MB 519.9 kB/s eta 0:00:19
   - -------------------------------------- 0.3/9.9 MB 550.1 kB/s eta 0:00:18
   - ---


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
!pip list

Package                   Version
------------------------- -----------
anyio                     4.13.0
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.3.0
attrs                     26.1.0
babel                     2.18.0
beautifulsoup4            4.14.3
bleach                    6.3.0
blinker                   1.9.0
certifi                   2026.5.20
cffi                      2.0.0
charset-normalizer        3.4.7
click                     8.4.1
colorama                  0.4.6
comm                      0.2.3
contourpy                 1.3.3
cycler                    0.12.1
dash                      4.1.0
debugpy                   1.8.20
decorator                 5.3.1
defusedxml                0.7.1
executing                 2.2.1
fastjsonschema            2.21.2
findspark                 2.0.1
Flask                     3.1.3
fonttools                 4.63.0
fqdn              


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
!pip install "dash[cloud]"

   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/50.6 kB ? eta -:--:--
   -------- --------


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [51]:
!pip uninstall plotly_cloud -y

In [55]:
!pip uninstall plotly-cloud -y

In [56]:
!pip install --upgrade dash plotly


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [58]:
!pip install --upgrade dash


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


[2026-05-31 14:40:42,700] ERROR in app: Exception on /favicon.ico [GET]
Traceback (most recent call last):
  File "d:\bottleneck-miner\venv\Lib\site-packages\flask\app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\bottleneck-miner\venv\Lib\site-packages\flask\app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\bottleneck-miner\venv\Lib\site-packages\dash\_get_app.py", line 17, in wrap
    return ctx.run(func, self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\bottleneck-miner\venv\Lib\site-packages\dash\dash.py", line 1250, in index
    scripts = self._generate_scripts_html()
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\bottleneck-miner\venv\Lib\site-packages\dash\dash.py", line 1151, in _generat